In [3]:
import requests
import us
from collections import Counter
from flair.data import Sentence

from flair.nn import Classifier
HEADERS = {"User-Agent": "WildfireKG/1.0"}
SPARQL = "https://query.wikidata.org/sparql"
tagger = Classifier.load('ner')

/Users/meenuravi/miniconda3/envs/flamedvisor/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


2026-04-19 10:39:44,381 SequenceTagger predicts: Dictionary with 20 tags: <unk>, O, S-ORG, S-MISC, B-PER, E-PER, S-LOC, B-ORG, E-ORG, I-PER, S-PER, B-MISC, I-MISC, E-MISC, I-ORG, B-LOC, E-LOC, I-LOC, <START>, <STOP>


In [15]:
STATE_TO_REGION = {
    "Connecticut": "Northeast", "Maine": "Northeast", "Massachusetts": "Northeast",
    "New Hampshire": "Northeast", "Rhode Island": "Northeast", "Vermont": "Northeast",
    "New Jersey": "Northeast", "New York": "Northeast", "Pennsylvania": "Northeast",
    "Illinois": "Midwest", "Indiana": "Midwest", "Michigan": "Midwest",
    "Ohio": "Midwest", "Wisconsin": "Midwest", "Iowa": "Midwest",
    "Kansas": "Midwest", "Minnesota": "Midwest", "Missouri": "Midwest",
    "Nebraska": "Midwest", "North Dakota": "Midwest", "South Dakota": "Midwest",
    "Delaware": "South", "Florida": "South", "Georgia": "South",
    "Maryland": "South", "North Carolina": "South", "South Carolina": "South",
    "Virginia": "South", "Washington DC": "South", "West Virginia": "South",
    "Alabama": "South", "Kentucky": "South", "Mississippi": "South",
    "Tennessee": "South", "Arkansas": "South", "Louisiana": "South",
    "Oklahoma": "South", "Texas": "South",
    "Arizona": "West", "Colorado": "West", "Idaho": "West", "Montana": "West",
    "Nevada": "West", "New Mexico": "West", "Utah": "West", "Wyoming": "West",
    "Alaska": "West", "California": "West", "Hawaii": "West",
    "Oregon": "West", "Washington": "West",
}

def get_region(hierarchy):
    state_names = {s.name for s in us.STATES}
    for h in hierarchy:
        if h in state_names:
            return STATE_TO_REGION.get(h)
    return None

In [55]:

def flair_loc(sentence):
    locs = []
    for entity in sentence.get_spans("ner"):
        label = entity.get_label("ner")
        if label.value in ("LOC", "GPE") and label.score >= 0.85:
            locs.append(entity.text)
    return list(set(locs))

def sort_hierarchy(hierarchy):
    state_names = {s.name for s in us.STATES}

    counties = [h for h in hierarchy if any(k in h.lower() for k in ["county", "parish", "borough"])]
    if counties:
        county = counties[0]
        matched_state = None
        for h in hierarchy:
            if h in state_names:
                r = requests.get(SPARQL, params={"query": f"""
                    ASK {{ ?c rdfs:label "{county}"@en ; wdt:P131 ?s . ?s rdfs:label "{h}"@en . }}
                """, "format": "json"}, headers=HEADERS, timeout=10)
                try:
                    if r.json().get("boolean"):
                        matched_state = h
                        break
                except Exception:
                    pass
        if matched_state:
            hierarchy = [h for h in hierarchy if h not in state_names or h == matched_state]

    def level(h):
        h_lower = h.lower()
        if "united states" in h_lower:
            return 6
        if h in STATE_TO_REGION.values():
            return 5
        if h in state_names:
            return 4
        if any(k in h_lower for k in ["national park", "national forest", "wilderness", "reserve"]):
            return 1
        if any(k in h_lower for k in ["county", "parish", "borough"]):
            return 2
        return 0

    # insert region for any state in hierarchy
    for h in list(hierarchy):
        if h in state_names:
            region = STATE_TO_REGION.get(h)
            if region and region not in hierarchy:
                hierarchy.append(region)

    return sorted(hierarchy, key=level)

def search_candidates(query):
    r = requests.get("https://www.wikidata.org/w/api.php", params={
        "action": "wbsearchentities", "format": "json",
        "language": "en", "search": query, "type": "item", "limit": 10
    }, headers=HEADERS)
    return [c["id"] for c in r.json().get("search", [])]

def get_hierarchy_for_qid(qid):
    query = f"""
    SELECT DISTINCT ?parent ?parentLabel WHERE {{
      {{ wd:{qid} wdt:P131+ ?parent . }}
      UNION
      {{ wd:{qid} wdt:P17 ?parent . }}
      SERVICE wikibase:label {{ bd:serviceParam wikibase:language "en". }}
    }}
    """
    r = requests.get(SPARQL, params={"query": query, "format": "json"}, headers=HEADERS, timeout=30)
    rows = r.json().get("results", {}).get("bindings", [])
    hierarchy = list(dict.fromkeys(
        row["parentLabel"]["value"] for row in rows if "parentLabel" in row
    ))
    label_r = requests.get(
        "https://www.wikidata.org/w/api.php",
        params={"action": "wbgetentities", "format": "json",
                "ids": qid, "props": "labels", "languages": "en"},
        headers=HEADERS
    ).json()
    place_label = label_r["entities"][qid]["labels"].get("en", {}).get("value")
    if place_label and place_label not in hierarchy:
        hierarchy.insert(0, place_label)
    return sort_hierarchy(hierarchy)

def resolve_by_prominence(candidates, method="population"):
    values = " ".join(f"wd:{q}" for q in candidates)
    if method == "population":
        query = f"""
        SELECT ?item (COALESCE(?pop, 0) AS ?score) WHERE {{
          VALUES ?item {{ {values} }}
          OPTIONAL {{ ?item wdt:P1082 ?pop . }}
          ?item wdt:P17 wd:Q30 .
        }} ORDER BY DESC(?score) LIMIT 1
        """
    else:
        query = f"""
        SELECT ?item (COUNT(?sitelink) AS ?score) WHERE {{
          VALUES ?item {{ {values} }}
          ?sitelink schema:about ?item .
          ?item wdt:P17 wd:Q30 .
        }} GROUP BY ?item ORDER BY DESC(?score) LIMIT 1
        """
    r = requests.get(SPARQL, params={"query": query, "format": "json"}, headers=HEADERS, timeout=30)
    rows = r.json().get("results", {}).get("bindings", [])
    return rows[0]["item"]["value"].split("/")[-1] if rows else None
def search_cdp(place, state):
    query = f"""
    SELECT ?item WHERE {{
      ?item rdfs:label "{place}"@en .
      ?item wdt:P131+ ?s .
      ?s rdfs:label "{state}"@en .
    }} LIMIT 5
    """
    r = requests.get(SPARQL, params={"query": query, "format": "json"}, headers=HEADERS, timeout=30)
    # print(r.status)
    rows = r.json().get("results", {}).get("bindings", [])
    return rows[0]["item"]["value"].split("/")[-1] if rows else None
def get_hierarchy(place: str, state: str = None, context_text: str = None, tagger=tagger) -> dict:
    def build_output(qid, hierarchy, disambiguation):
        p31_query = f"""
        SELECT ?typeLabel ?isNatural WHERE {{
          wd:{qid} wdt:P31 ?type .
          BIND(EXISTS {{
            {{ ?type wdt:P279* wd:Q1076486 }}  # natural geographic object
            UNION
            {{ ?type wdt:P279* wd:Q473972 }}   # protected area (national parks)
            UNION
            {{ ?type wdt:P279* wd:Q1protected }} 
          }} AS ?isNatural)
          SERVICE wikibase:label {{ bd:serviceParam wikibase:language "en". }}
        }}
        """
        r = requests.get(SPARQL, params={"query": p31_query, "format": "json"}, headers=HEADERS, timeout=30)
        rows = r.json().get("results", {}).get("bindings", [])
        types = [row["typeLabel"]["value"].lower() for row in rows]
        is_natural = any(row.get("isNatural", {}).get("value") == "true" for row in rows)

        return {
            "place": place,
            "qid": qid,
            "hierarchy": hierarchy,
            "geo_scope": hierarchy[0] if hierarchy else place,
            "scope_type": types[0].replace("in the united states","") if types else None,
            "is_natural": is_natural,
            "is_us": "United States" in hierarchy,
            "disambiguation": disambiguation,
        }

    # Tier 1: explicit state
    if state:
        candidates = search_candidates(f"{place}, {state}")
        print(candidates)
        if candidates:
            for qid in candidates:
                hierarchy = get_hierarchy_for_qid(qid)
                if state in hierarchy:
                    return build_output(qid, hierarchy, "explicit_state")
            # Tier 1b: CDP fallback
        else:
            qid = search_cdp(place, state)
            if qid:
                return build_output(qid, get_hierarchy_for_qid(qid), "cdp")
        return {"place": place, "error": "Could not resolve"}

    # Tier 2: extract state from context via Flair
    if context_text and tagger:
        sentence = Sentence(context_text)
        tagger.predict(sentence)
        locs = flair_loc(sentence)
        states_found = [us.states.lookup(loc).name for loc in locs if us.states.lookup(loc)]
        if states_found:
            best_state = Counter(states_found).most_common(1)[0][0]
            candidates = search_candidates(f"{place}, {best_state}")
            if candidates:
                hierarchy = get_hierarchy_for_qid(candidates[0])
                if "United States" in hierarchy:
                    return build_output(candidates[0], hierarchy, "context")

    # Tier 3 & 4: prominence-based
    candidates = search_candidates(place)
    if not candidates:
        return {"place": place, "error": "Not found"}

    for method in ("population", "sitelinks"):
        qid = resolve_by_prominence(candidates, method)
        if qid:
            hierarchy = get_hierarchy_for_qid(qid)
            if hierarchy:
                return build_output(qid, hierarchy, method)

    return {"place": place, "error": "Could not resolve"}

In [56]:
# Tier 1 — explicit state
print(get_hierarchy("East coast"))

# # Tier 2 — context
# print(get_hierarchy("Springfield", context_text="The wildfire spread across Illinois counties..."))

# # # Tier 3 — no hints, fully automatic
# print(get_hierarchy("Springfield"))

# # Regular city
# print(get_hierarchy("Los Angeles"))

# # County
# print(get_hierarchy("Cook County"))

{'place': 'East coast', 'qid': 'Q4268', 'hierarchy': ['East Coast of the United States', 'United States'], 'geo_scope': 'East Coast of the United States', 'scope_type': 'coast', 'is_natural': False, 'is_us': True, 'disambiguation': 'population'}


In [54]:
def get_hierarchy_for_qid(qid):
    query = f"""
    SELECT DISTINCT ?parent ?parentLabel WHERE {{
      {{ wd:{qid} wdt:P131+ ?parent . }}
      UNION
      {{ wd:{qid} wdt:P17 ?parent . }}
      SERVICE wikibase:label {{ bd:serviceParam wikibase:language "en". }}
    }}
    """

In [57]:
print(get_hierarchy("Yakama Indian Reservation"))

{'place': 'Yakama Indian Reservation', 'qid': 'Q3457242', 'hierarchy': ['Yakama Indian Reservation', 'Yakima County', 'Klickitat County', 'Washington', 'West', 'United States'], 'geo_scope': 'Yakama Indian Reservation', 'scope_type': 'indian reservation of the united states', 'is_natural': False, 'is_us': True, 'disambiguation': 'population'}
